# C09_E02 — PID forma paralelo vs ISA

**Capítulo:** 9 — Control PID en la práctica industrial  
**Identificador:** `C09_E02`  
**Objetivo pedagógico:** Demostrar la equivalencia entre las formas y la importancia de traducir parámetros.  
**Librerías:** python-control, numpy, matplotlib

## Ejemplo industrial

Bloque PID de fabricante: distinción de forma para cargar parámetros correctamente.

---

> **Aviso de uso responsable.** Este notebook es didáctico. No es código de producción. Cualquier implementación real requiere validación independiente. Ver `docs/politica_uso_responsable.md`.

## 1. Forma paralelo vs forma ISA

In [1]:
import control as ct
import numpy as np
import matplotlib.pyplot as plt
import os

# Planta común
G = ct.tf([1.0], [10.0, 1.0])

# Forma paralelo: u = Kp*e + Ki*int(e) + Kd*de/dt
Kp, Ki, Kd = 0.4, 0.04, 0.0
PI_paralelo = ct.tf([Kp, Ki], [1, 0])

# Forma ISA: u = Kp_isa * (e + (1/Ti)*int(e) + Td*de/dt)
# Con Ti = Kp/Ki  →  Kp_isa = Kp, Ti = Kp/Ki
Kp_isa = Kp; Ti = Kp/Ki
PI_isa = ct.tf([Kp_isa*Ti, Kp_isa], [Ti, 0])

print("Paralelo num/den:", PI_paralelo.num[0][0], PI_paralelo.den[0][0])
print("ISA num/den:     ", PI_isa.num[0][0], PI_isa.den[0][0])

Paralelo num/den: [0.4  0.04] [1 0]
ISA num/den:      [4.  0.4] [10.  0.]


## 2. Respuesta a escalón en lazo cerrado

In [2]:
sys_par = ct.feedback(PI_paralelo*G, 1)
sys_isa = ct.feedback(PI_isa*G, 1)
t = np.linspace(0, 100, 500)
_, y_par = ct.step_response(sys_par, t)
_, y_isa = ct.step_response(sys_isa, t)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t, y_par, label="Forma paralelo")
ax.plot(t, y_isa, '--', label="Forma ISA")
ax.set_xlabel("t (s)"); ax.set_ylabel("y(t)")
ax.legend(); ax.grid(alpha=0.3)
ax.set_title("C09_E02 — Paralelo vs ISA (parámetros equivalentes)")
out = '../../figures/cap09/fig_C09_F02_paralelo_vs_isa.png'
os.makedirs(os.path.dirname(out), exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=120); plt.show()

## 3. Interpretación

Las dos formas producen el mismo controlador cuando los parámetros están correctamente traducidos (`Ti = Kp/Ki`). En la práctica industrial, el peligro está en aplicar parámetros de una forma a otra sin traducir: leer el manual del fabricante (Honeywell, Emerson, Siemens, Allen-Bradley) antes de cargar valores en bloque PID.